In [1]:
print("hello world")

hello world


In [2]:
# Pandas library import karein aur usay "pd" naam se use karein
import pandas as pd


In [14]:
import re
from datetime import datetime
from pathlib import Path
from urllib.error import HTTPError

# Paste the full Google Sheets URL here; the current sheet link is already included
sheet_url = "https://docs.google.com/spreadsheets/d/1Loa93V_Y782f9z8bq2NmFbotj9iVzKOxPZ93wk4grfs"

# Find the project root whether the notebook runs from the project folder or pipeline folder
project_root = Path.cwd()
if not (project_root / "raw").is_dir() and (project_root.parent / "raw").is_dir():
    project_root = project_root.parent

# Create the raw and error folders if they do not already exist
raw_folder = project_root / "raw"
error_folder = project_root / "error"
raw_folder.mkdir(parents=True, exist_ok=True)
error_folder.mkdir(parents=True, exist_ok=True)

# Give this run a unique filename so an older CSV is never accidentally moved
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = raw_folder / f"google_sheet_{run_id}.csv"
temp_csv_path = raw_folder / f"google_sheet_{run_id}.in_progress.csv"

try:
    # Extract the spreadsheet ID from the Google Sheets URL
    sheet_match = re.search(r"/spreadsheets/d/([a-zA-Z0-9_-]+)", sheet_url)
    if sheet_match is None:
        raise ValueError("The Google Sheets URL is not valid. Paste the full sheet URL.")
    sheet_id = sheet_match.group(1)

    # Use the tab gid in the URL, or select the first tab if no gid is included
    gid_match = re.search(r"[?&#]gid=(\d+)", sheet_url)
    gid = gid_match.group(1) if gid_match else "0"

    # Build the CSV export URL and load the sheet into the DataFrame
    csv_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
    df = pd.read_csv(csv_url)

    # Add a 1-based index for every data row as the first column
    df.insert(0, "row_index", range(1, len(df) + 1))

    # Write to a temporary CSV first so an incomplete file is not left in raw
    df.to_csv(temp_csv_path, index=False)

    # Move the completed CSV into its final location inside the raw folder
    temp_csv_path.replace(csv_path)

    # Display the DataFrame after the CSV has been saved successfully
    display(df)
    print(f"CSV saved successfully: {csv_path}")

except Exception as error:
    # Move a partial or newly-created CSV to error if this run produced one
    failed_csv_path = error_folder / f"google_sheet_{run_id}_failed.csv"
    if temp_csv_path.exists():
        temp_csv_path.replace(failed_csv_path)
    elif csv_path.exists():
        csv_path.replace(failed_csv_path)

    # Save the exception details because a sheet-read error may happen before any CSV exists
    error_log_path = error_folder / f"google_sheet_{run_id}_error.txt"
    error_log_path.write_text(f"{type(error).__name__}: {error}\n", encoding="utf-8")

    # Explain the common Google Sheets 404 cause while preserving other errors
    if isinstance(error, HTTPError) and error.code == 404:
        print("Google returned 404. Check the sheet URL/tab gid and confirm the sheet is shared for reading.")
    print(f"Failure details saved to: {error_log_path}")
    raise

,row_index,Order ID,customer_id,Customer Name,Gender,Age,Email,Phone,Location,Order Date,...,Category,Qty,Unit Price,Discount,Payment Method,Order Status,Is Prime Member?,Rating,Weight,Notes
0,1,ORD-20000,C1118,Mr./Ms. Hira Sheikh,f,70,hira.sheikh48@outlook.com,0382-9114678,Quetta,6-Aug-24,...,Apparel,1,"7,242",5%,JazzCash,Shipped,no,?,9572 g,NaN
1,2,ORD-20001,C1265,Mr./Ms. David Khan,F,27 yrs,david.khan87@yahoo.com,(3588) 916375,multan - Punjab,16/08/2023,...,ELECTRONICS,1,63141.3,10%,CC,returned,No,NaN,10.9 kg,NaN
2,3,ORD-20002,C1334,Maria Chaudhry,female,42,maria.chaudhry77@yahoo.com,9.23E+11,Islmabad - ICT,20230225,...,Appliances,two,7335.9,20%,bank transfer,dlvd,FALSE,NaN,1.4 kg,leave at door
3,4,ORD-20003,C1126,sophia khan,Female,36,sophia.khan51@outlook.com,3339500479,LAHORE,8/12/2024,...,Furnitue,100,15686.88,15,Wallet,Delivred,Y,NaN,10982 g,leave at door
4,5,ORD-20004,C1108,sophia khan,male,40,sophia.khan50@hotmail.com,9.23E+11,"Multan , Punjab",29/09/2024,...,FURNITURE,3,"Rs. 18,251",NaN,creditcard,Shipped,yes,?,3.62,Gift wrap please
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206,207,ORD-20194,C1069,hira johnson,FEMALE,38,hira.johnson25@hotmail.com,3210470952,Multan/Punjab,23-Aug-24,...,Clothing,?,1190.21,0,Cash,PENDING,No,NaN,NaN,NaN
207,208,ORD-20195,C1156,James Garcia,MALE,200,james.garcia72yahoo.com,9.24E+11,Multan - Punjab,10/7/2023,...,Furniture,1,2461.7/-,0%,bank transfer,Shipped,Y,NaN,8.38,URGENT!!
208,209,ORD-20196,C1033,John Chaudhry,MALE,58,john.chaudhry86@hotmail.com,-8687564,Peshawar - KPK,19-Mar-23,...,electronics,1,56186.5/-,0,IBFT,cancelled,no,NaN,NaN,NaN
209,210,ORD-20197,C1226,Mr./Ms. James Miller,Female,69,james.miller49@hotmail.com,92 342 2096902,"Karachi, Sindh",27.09.2023,...,Electronics,1,55125.14,0.05,cod,Return,FALSE,?,9892 g,"Fragile, handle with care"


CSV saved successfully: d:\order\raw\google_sheet_20261020_062539.csv


In [3]:
from __future__ import annotations

import re
from pathlib import Path

from IPython.display import display
import pandas as pd


def normalize_column_name(column_name: str) -> str:
    cleaned = str(column_name).strip()
    cleaned = re.sub(r"[^a-zA-Z0-9]+", "_", cleaned)
    cleaned = cleaned.strip("_")
    return cleaned.lower()


def safe_text(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    if text.lower() in {"", "na", "n/a", "nan", "none", "null", "?", "-"}:
        return None
    return text


def normalize_customer_name(value):
    text = safe_text(value)
    if text is None:
        return None
    text = re.sub(r"(?i)\bmr\.?/ms\.?", "Mr./Ms.", text)
    text = re.sub(r"(?i)\bms\b", "Ms", text)
    text = re.sub(r"(?i)\bmr\b", "Mr", text)
    parts = re.split(r"\s+", text.strip())
    normalized_parts = []
    for part in parts:
        if not part:
            continue
        if "/" in part:
            normalized_parts.append(part)
        elif part.lower() in {"mr", "ms"}:
            normalized_parts.append(part)
        else:
            normalized_parts.append(part.capitalize())
    return " ".join(normalized_parts)


def normalize_gender(value):
    text = (safe_text(value) or "").lower()
    if text in {"m", "male", "man", "boy"}:
        return "M"
    if text in {"f", "female", "woman", "girl"}:
        return "F"
    return None


def normalize_email(value):
    text = safe_text(value)
    if text is None:
        return None
    return text.lower()


def normalize_phone(value):
    text = safe_text(value)
    if text is None:
        return None
    digits = re.sub(r"\D+", "", text)
    if len(digits) >= 10:
        return digits
    return None


def normalize_order_id(value):
    text = safe_text(value)
    if text is None:
        return None
    return str(text).upper()


def normalize_customer_id(value):
    text = safe_text(value)
    if text is None:
        return None
    return str(text).upper()


def normalize_location(value):
    text = safe_text(value)
    if text is None:
        return None

    text = text.replace("/", " - ")
    text = text.replace(",", " - ")
    text = re.sub(r"\s*[-–—]\s*", " - ", text)
    text = re.sub(r"\s+", " ", text).strip()

    city_map = {
        "quetta": "Quetta",
        "multan": "Multan",
        "islamabad": "Islamabad",
        "lahore": "Lahore",
        "karachi": "Karachi",
        "peshawar": "Peshawar",
        "faisalabad": "Faisalabad",
    }
    province_map = {
        "balochistan": "Balochistan",
        "punjab": "Punjab",
        "ict": "ICT",
        "kpk": "KPK",
        "sindh": "Sindh",
    }

    tokens = [token.strip() for token in text.split("-") if token.strip()]
    if len(tokens) == 2:
        city = city_map.get(tokens[0].lower(), tokens[0].title())
        province = province_map.get(tokens[1].lower(), tokens[1].title())
        return f"{city}, {province}"

    lower_text = text.lower()
    for key, value_name in city_map.items():
        if lower_text.startswith(key):
            return value_name
    return text.title()


def normalize_date(value):
    text = safe_text(value)
    if text is None:
        return None

    text = text.strip()
    if re.fullmatch(r"\d{8}", text):
        return pd.to_datetime(text, format="%Y%m%d").strftime("%Y-%m-%d")

    try:
        parsed = pd.to_datetime(text, errors="raise", infer_datetime_format=True)
        return parsed.strftime("%Y-%m-%d")
    except Exception:
        for fmt in (
            "%d-%b-%y",
            "%d-%b-%Y",
            "%d/%m/%Y",
            "%d/%m/%y",
            "%m/%d/%Y",
            "%m/%d/%y",
            "%Y/%m/%d",
            "%d.%m.%Y",
            "%d.%m.%y",
            "%d-%m-%Y",
            "%d-%m-%y",
            "%Y-%m-%d",
        ):
            try:
                parsed = pd.to_datetime(text, format=fmt, errors="raise")
                return parsed.strftime("%Y-%m-%d")
            except Exception:
                continue

    return None


def normalize_product(value):
    text = safe_text(value)
    if text is None:
        return None
    lowered = text.lower()
    replacements = {
        "smartphone": "Smartphone",
        "office chair": "Office Chair",
        "blender": "Blender",
        "sneakers": "Sneakers",
        "laptop": "Laptop",
        "phone": "Phone",
        "watch": "Watch",
        "headphones": "Headphones",
        "table": "Table",
        "chair": "Chair",
    }
    for key, replacement in replacements.items():
        if lowered == key or lowered.startswith(key):
            return replacement
    return text.title()


def normalize_category(value):
    text = safe_text(value)
    if text is None:
        return None
    lowered = text.lower().replace("_", " ")
    mapping = {
        "apparel": "Apparel",
        "clothing": "Apparel",
        "clothes": "Apparel",
        "electronics": "Electronics",
        "electronic": "Electronics",
        "elec.": "Electronics",
        "appliances": "Home Appliances",
        "home appliances": "Home Appliances",
        "homeappliances": "Home Appliances",
        "furnitue": "Furniture",
        "furniture": "Furniture",
    }
    return mapping.get(lowered, text.title())


def normalize_qty(value):
    text = safe_text(value)
    if text is None:
        return None
    lowered = text.lower()
    word_map = {
        "zero": 0,
        "one": 1,
        "two": 2,
        "three": 3,
        "four": 4,
        "five": 5,
        "six": 6,
        "seven": 7,
        "eight": 8,
        "nine": 9,
        "ten": 10,
    }
    if lowered in word_map:
        return int(word_map[lowered])
    match = re.search(r"-?\d+", lowered)
    if match:
        qty = int(match.group(0))
        return qty if qty >= 0 else None
    return None


def normalize_unit_price(value):
    text = safe_text(value)
    if text is None:
        return None
    cleaned = re.sub(r"(?i)\b(?:rs|pkr)\.?\b", "", text)
    cleaned = cleaned.replace(",", "")
    cleaned = cleaned.replace(" ", "")
    cleaned = re.sub(r"[^0-9.\-]", "", cleaned)
    if cleaned.startswith("."):
        cleaned = cleaned[1:]
    if cleaned.count(".") > 1:
        cleaned = cleaned.replace(".", "", cleaned.count(".") - 1)
    if cleaned in {"", ".", "-", "-."}:
        return None
    try:
        return float(cleaned)
    except ValueError:
        return None


def normalize_discount(value):
    text = safe_text(value)
    if text is None:
        return None
    cleaned = text.strip()
    if re.search(r"%", cleaned):
        numeric = re.sub(r"[^0-9.\-]", "", cleaned)
        if not numeric:
            return None
        percentage = float(numeric)
        return percentage
    numeric = re.sub(r"[^0-9.\-]", "", cleaned)
    if not numeric:
        return None
    decimal_value = float(numeric)
    if decimal_value <= 1:
        return decimal_value * 100
    return decimal_value


def normalize_payment_method(value):
    text = safe_text(value)
    if text is None:
        return None
    lowered = text.lower().replace("_", " ")
    replacements = {
        "bank transfer": "Bank Transfer",
        "bank": "Bank",
        "creditcard": "Credit Card",
        "credit card": "Credit Card",
        "cc": "Credit Card",
        "cod": "Cash on Delivery",
        "cash on delivery": "Cash on Delivery",
        "wallet": "Wallet",
        "jazzcash": "JazzCash",
        "easypaisa": "Easypaisa",
        "cash": "Cash",
        "ibft": "IBFT",
        "card": "Card",
    }
    return replacements.get(lowered, text.title())


def normalize_order_status(value):
    text = safe_text(value)
    if text is None:
        return None
    lowered = text.lower()
    mapping = {
        "shipped": "Shipped",
        "delivered": "Delivered",
        "delivred": "Delivered",
        "dlvd": "Delivered",
        "processing": "Processing",
        "pending": "Pending",
        "returned": "Returned",
        "return": "Returned",
        "cancelled": "Cancelled",
        "canceled": "Canceled",
        "in transit": "In Transit",
        "completed": "Completed",
    }
    return mapping.get(lowered, text.title())


def normalize_prime_member(value):
    text = safe_text(value)
    if text is None:
        return None
    lowered = text.lower()
    if lowered in {"true", "t", "y", "yes", "1"}:
        return True
    if lowered in {"false", "f", "n", "no", "0"}:
        return False
    return None


def normalize_rating(value):
    text = safe_text(value)
    if text is None:
        return None
    lowered = text.lower()
    if lowered in {"?", "-", "na", "n/a"}:
        return None
    if "star" in lowered:
        match = re.search(r"(\d+(?:\.\d+)?)", lowered)
        if match:
            return float(match.group(1))
        return None
    numeric = re.sub(r"[^0-9.\-]", "", text)
    if numeric == "":
        return None
    try:
        rating = float(numeric)
        return rating if 0 <= rating <= 5 else None
    except ValueError:
        return None


def normalize_weight(value):
    text = safe_text(value)
    if text is None:
        return None
    lowered = text.lower()
    match = re.search(r"([0-9]+(?:\.[0-9]+)?)\s*(g|kg|lbs|lb)", lowered)
    if not match:
        numeric = re.search(r"([0-9]+(?:\.[0-9]+)?)", lowered)
        if numeric:
            return float(numeric.group(1))
        return None
    amount = float(match.group(1))
    unit = match.group(2)
    if unit == "g":
        return round(amount / 1000, 3)
    if unit in {"kg", "kilogram"}:
        return round(amount, 3)
    if unit in {"lb", "lbs"}:
        return round(amount * 0.453592, 3)
    return round(amount, 3)


def transform_orders(dataframe: pd.DataFrame) -> pd.DataFrame:
    df = dataframe.copy()
    df.columns = [normalize_column_name(col) for col in df.columns]

    df["row_index"] = df.get("row_index").apply(lambda x: int(float(x)) if pd.notna(x) and str(x).strip() not in {"", "nan"} else None)
    df["order_id"] = df.get("order_id").apply(normalize_order_id)
    df["customer_id"] = df.get("customer_id").apply(normalize_customer_id)
    df["customer_name"] = df.get("customer_name").apply(normalize_customer_name)
    df["gender"] = df.get("gender").apply(normalize_gender)
    df["age"] = df.get("age").apply(lambda x: pd.to_numeric(re.search(r"\d+", str(x) or "") .group(0), errors="coerce") if (safe_text(x) and re.search(r"\d+", str(x))) else None)
    df["age"] = df["age"].where(df["age"].between(18, 90), None)
    df["email"] = df.get("email").apply(normalize_email)
    df["phone"] = df.get("phone").apply(normalize_phone)
    df["location"] = df.get("location").apply(normalize_location)
    df["order_date"] = df.get("order_date").apply(normalize_date)
    df["ship_date"] = df.get("shipdate").apply(normalize_date)
    df["product"] = df.get("product").apply(normalize_product)
    df["category"] = df.get("category").apply(normalize_category)
    df["qty"] = df.get("qty").apply(normalize_qty)
    df["qty"] = df["qty"].where(df["qty"].between(0, 1000), None)
    df["unit_price"] = df.get("unit_price").apply(normalize_unit_price)
    df["discount"] = df.get("discount").apply(normalize_discount)
    df["payment_method"] = df.get("payment_method").apply(normalize_payment_method)
    df["order_status"] = df.get("order_status").apply(normalize_order_status)
    df["is_prime_member"] = df.get("is_prime_member").apply(normalize_prime_member)
    df["rating"] = df.get("rating").apply(normalize_rating)
    df["weight_kg"] = df.get("weight").apply(normalize_weight)
    df["notes"] = df.get("notes").apply(lambda x: safe_text(x) or "")

    desired_columns = [
        "row_index",
        "order_id",
        "customer_id",
        "customer_name",
        "gender",
        "age",
        "email",
        "phone",
        "location",
        "order_date",
        "ship_date",
        "product",
        "category",
        "qty",
        "unit_price",
        "discount",
        "payment_method",
        "order_status",
        "is_prime_member",
        "rating",
        "weight_kg",
        "notes",
    ]

    # Preserve the original row order but keep only the normalized columns.
    cleaned = df.reindex(columns=desired_columns)

    cleaned["order_id"] = cleaned["order_id"].astype("string")
    cleaned["customer_id"] = cleaned["customer_id"].astype("string")
    cleaned["customer_name"] = cleaned["customer_name"].astype("string")
    cleaned["gender"] = cleaned["gender"].astype("string")
    cleaned["email"] = cleaned["email"].astype("string")
    cleaned["phone"] = cleaned["phone"].astype("string")
    cleaned["location"] = cleaned["location"].astype("string")
    cleaned["product"] = cleaned["product"].astype("string")
    cleaned["category"] = cleaned["category"].astype("string")
    cleaned["payment_method"] = cleaned["payment_method"].astype("string")
    cleaned["order_status"] = cleaned["order_status"].astype("string")
    cleaned["notes"] = cleaned["notes"].astype("string")

    cleaned["age"] = pd.to_numeric(cleaned["age"], errors="coerce").astype("Int64")
    cleaned["qty"] = pd.to_numeric(cleaned["qty"], errors="coerce").astype("Int64")
    cleaned["unit_price"] = pd.to_numeric(cleaned["unit_price"], errors="coerce")
    cleaned["discount"] = pd.to_numeric(cleaned["discount"], errors="coerce")
    cleaned["rating"] = pd.to_numeric(cleaned["rating"], errors="coerce")
    cleaned["weight_kg"] = pd.to_numeric(cleaned["weight_kg"], errors="coerce")
    cleaned["is_prime_member"] = cleaned["is_prime_member"].astype("boolean")
    return cleaned


def main() -> pd.DataFrame:
    project_root = Path.cwd()
    if not (project_root / "raw").is_dir() and (project_root.parent / "raw").is_dir():
        project_root = project_root.parent
    raw_dir = project_root / "raw"
    target_dir = project_root / "transformation"
    target_dir.mkdir(parents=True, exist_ok=True)

    csv_files = sorted(raw_dir.glob("*.csv"), key=lambda path: path.stat().st_mtime, reverse=True)
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {raw_dir}")

    raw_path = csv_files[0]
    df = pd.read_csv(raw_path, dtype=str)
    df = df.replace({"nan": None, "NaN": None, "None": None, "": None})

    cleaned = transform_orders(df)

    output_name = f"stg_{raw_path.stem}.csv"
    output_path = target_dir / output_name
    cleaned.to_csv(output_path, index=False)

    generic_path = target_dir / "stg_orders.csv"
    cleaned.to_csv(generic_path, index=False)

    print(f"Saved cleaned data to {output_path}")
    print(f"Saved generic staging output to {generic_path}")
    print(f"Transformed {len(cleaned)} order rows from {raw_path.name}")
    display(cleaned)
    return cleaned


df = main()


Saved cleaned data to d:\order\transformation\stg_google_sheet_20261020_062539.csv
Saved generic staging output to d:\order\transformation\stg_orders.csv
Transformed 211 order rows from google_sheet_20261020_062539.csv


,row_index,order_id,customer_id,customer_name,gender,age,email,phone,location,order_date,...,category,qty,unit_price,discount,payment_method,order_status,is_prime_member,rating,weight_kg,notes
0,1,ORD-20000,C1118,Mr./Ms. Hira Sheikh,F,70,hira.sheikh48@outlook.com,03829114678,Quetta,2024-08-06,...,Apparel,1,7242.00,5.0,JazzCash,Shipped,False,NaN,9.572,
1,2,ORD-20001,C1265,Mr./Ms. David Khan,F,27,david.khan87@yahoo.com,3588916375,"Multan, Punjab",2023-08-16,...,Electronics,1,63141.30,10.0,Credit Card,Returned,False,NaN,10.900,
2,3,ORD-20002,C1334,Maria Chaudhry,F,42,maria.chaudhry77@yahoo.com,<NA>,"Islmabad, ICT",2023-02-25,...,Home Appliances,2,7335.90,20.0,Bank Transfer,Delivered,False,NaN,1.400,leave at door
3,4,ORD-20003,C1126,Sophia Khan,F,36,sophia.khan51@outlook.com,3339500479,Lahore,2024-12-08,...,Furniture,100,15686.88,15.0,Wallet,Delivered,True,NaN,10.982,leave at door
4,5,ORD-20004,C1108,Sophia Khan,M,40,sophia.khan50@hotmail.com,<NA>,"Multan, Punjab",2024-09-29,...,Furniture,3,18251.00,NaN,Credit Card,Shipped,True,NaN,3.620,Gift wrap please
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206,207,ORD-20194,C1069,Hira Johnson,F,38,hira.johnson25@hotmail.com,3210470952,"Multan, Punjab",2024-08-23,...,Apparel,<NA>,1190.21,0.0,Cash,Pending,False,NaN,NaN,
207,208,ORD-20195,C1156,James Garcia,M,<NA>,james.garcia72yahoo.com,<NA>,"Multan, Punjab",2023-07-10,...,Furniture,1,NaN,0.0,Bank Transfer,Shipped,True,NaN,8.380,URGENT!!
208,209,ORD-20196,C1033,John Chaudhry,M,58,john.chaudhry86@hotmail.com,<NA>,"Peshawar, KPK",2023-03-19,...,Electronics,1,NaN,0.0,IBFT,Cancelled,False,NaN,NaN,
209,210,ORD-20197,C1226,Mr./Ms. James Miller,F,69,james.miller49@hotmail.com,923422096902,"Karachi, Sindh",2023-09-27,...,Electronics,1,55125.14,5.0,Cash on Delivery,Returned,False,NaN,9.892,"Fragile, handle with care"


In [4]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd()
if not (project_root / "transformation").is_dir() and (project_root.parent / "transformation").is_dir():
    project_root = project_root.parent

df = pd.read_csv(project_root / "transformation" / "stg_orders.csv")
print("Rows and columns:", df.shape)
df.info()

NameError: name 'df' is not defined

In [ ]:
import json
from pathlib import Path

notebook_path = Path.cwd() / "pipeline" / "raw_to_stg_pythonfile.ipynb"
if not notebook_path.is_file():
    notebook_path = Path.cwd() / "raw_to_stg_pythonfile.ipynb"

with notebook_path.open(encoding="utf-8") as notebook_file:
    notebook = json.load(notebook_file)

transform_source = next(
    "".join(cell["source"])
    for cell in notebook["cells"]
    if cell["cell_type"] == "code" and "def transform_orders(" in "".join(cell["source"])
)
exec(compile(transform_source, str(notebook_path), "exec"), globals())

print("Rows and columns:", df.shape)
df.info()